## Project Data Science Lab 2 

In [2]:
import pandas as pd
import re
import os
import gensim
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.metrics.pairwise import cosine_similarity
from nltk.corpus import stopwords
import numpy as np
import nltk

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### Preprocessing

In [3]:
# Load the stopwords
stop_words = set(stopwords.words('english'))

# Function to preprocess the text without using NLTK
def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove special characters and digits
    text = re.sub(r'\W', ' ', text)
    text = re.sub(r'\d', ' ', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    # Tokenize the text (split by spaces)
    words = text.split()
    # Remove stopwords using the custom stopwords list
    words = [word for word in words if word not in stop_words]
    return words


In [4]:
# Apply preprocessing to the course details
file_path = '../annotated_course_details.xlsx'
df_courses = pd.read_excel(file_path)
# df['preprocessed_course_detail'] = df['course detail description'].apply(preprocess_text)
# df.head()
df_courses['preprocessed_course_detail'] = df_courses['course detail description'].apply(preprocess_text)
df_courses['preprocessed_task_phrases'] = df_courses['task_phrases'].apply(preprocess_text)
df_courses['preprocessed_skill_phrases'] = df_courses['skill_phrases'].apply(preprocess_text)
# df
# Combine annotated phrases with preprocessed course details
df_courses['combined_text_courses'] = (
    df_courses['preprocessed_course_detail'].apply(lambda x: " ".join(x)) + " " + 
    df_courses['preprocessed_task_phrases'].apply(lambda x: " ".join(x)) + " " + 
    df_courses['preprocessed_skill_phrases'].apply(lambda x: " ".join(x)))

In [5]:
# Load the skill roles dataset
df_skill_roles = pd.read_excel('../annotated_skill_roles.xlsx')

# Apply preprocessing to the mission, main tasks, and key skills columns
df_skill_roles['preprocessed_mission'] = df_skill_roles['mission'].apply(preprocess_text)
df_skill_roles['preprocessed_main_tasks'] = df_skill_roles['main_tasks'].apply(preprocess_text)
df_skill_roles['preprocessed_key_skills'] = df_skill_roles['key_skills'].apply(preprocess_text)
df_skill_roles['preprocessed_task_phrases'] = df_skill_roles['unique_task_phrases'].apply(preprocess_text)
df_skill_roles['preprocessed_skill_phrases'] = df_skill_roles['unique_skill_phrases'].apply(preprocess_text)

# Combine the preprocessed text and annotated phrases into a single column
df_skill_roles['combined_text_skill_roles'] = (
    df_skill_roles['preprocessed_mission'].apply(lambda x: " ".join(x)) + " " +
    df_skill_roles['preprocessed_main_tasks'].apply(lambda x: " ".join(x)) + " " +
    df_skill_roles['preprocessed_key_skills'].apply(lambda x: " ".join(x)) + " " +
    df_skill_roles['preprocessed_task_phrases'].apply(lambda x: " ".join(x)) + " " +
    df_skill_roles['preprocessed_skill_phrases'].apply(lambda x: " ".join(x))
)

In [8]:
import pandas as pd
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.metrics.pairwise import cosine_similarity
import time

# Load your datasets
# Extract the combined preprocessed text
courses_text = df_courses['combined_text_courses'].tolist()
skill_roles_text = df_skill_roles['combined_text_skill_roles'].tolist()

# Tag the documents (necessary for Doc2Vec)
tagged_courses = [TaggedDocument(words=text.split(), tags=[f'COURSE_{i}']) for i, text in enumerate(courses_text)]
tagged_skill_roles = [TaggedDocument(words=text.split(), tags=[f'SKILL_ROLE_{i}']) for i, text in enumerate(skill_roles_text)]
tagged_documents = tagged_courses + tagged_skill_roles

# Hyperparameter configurations to test
configs = [
    {"vector_size": 50, "window": 7, "epochs": 30, "dm": 0},
    {"vector_size": 50, "window": 10, "epochs": 30, "dm": 0},
    {"vector_size": 50, "window": 15, "epochs": 30, "dm": 0},
    {"vector_size": 50, "window": 20, "epochs": 30, "dm": 0},
    {"vector_size": 300, "window": 7, "epochs": 30, "dm": 0},
    {"vector_size": 300, "window": 10, "epochs": 30, "dm": 0},
    {"vector_size": 300, "window": 15, "epochs": 30, "dm": 0},
    {"vector_size": 300, "window": 20, "epochs": 30, "dm": 0},
]

# Store results
results = []

for config in configs:
    start_time = time.time()
    
    # Initialize the Doc2Vec model with the current configuration
    model = Doc2Vec(
        vector_size=config["vector_size"],
        window=config["window"],
        min_count=2,
        workers=4,
        epochs=config["epochs"],
        dm=config["dm"]
    )

    # Build the vocabulary and train the model
    model.build_vocab(tagged_documents)
    model.train(tagged_documents, total_examples=model.corpus_count, epochs=model.epochs)
    
    # Extract vectors for courses and skill roles
    course_vectors = [model.dv[f'COURSE_{i}'] for i in range(len(courses_text))]
    skill_role_vectors = [model.dv[f'SKILL_ROLE_{i}'] for i in range(len(skill_roles_text))]
    
    # Calculate the cosine similarity between each course and each skill role
    similarity_matrix = cosine_similarity(course_vectors, skill_role_vectors)
    
    # Calculate metrics
    avg_similarity = similarity_matrix.mean()
    negative_scores_percentage = (similarity_matrix < 0).mean() * 100
    training_time = time.time() - start_time
    
    # Store results
    results.append({
        "Vector Size": config["vector_size"],
        "Window": config["window"],
        "Epochs": config["epochs"],
        "Distributed Memory": "Yes" if config["dm"] == 1 else "No",
        "Average Similarity": avg_similarity,
        "Negative Scores (%)": negative_scores_percentage,
        "Training Time (s)": training_time
    })

# Convert the results to a DataFrame for easier analysis
results_df = pd.DataFrame(results)

# Display the results
results_df

# Save the results to an Excel file
results_df.to_excel('doc2vec_hyperparameter_comparison.xlsx', index=False)


In [9]:
results_df

,Vector Size,Window,Epochs,Distributed Memory,Average Similarity,Negative Scores (%),Training Time (s)
0,50,7,30,No,0.320380,0.226449,3.394999
1,50,10,30,No,0.322533,0.237772,3.441997
2,50,15,30,No,0.321294,0.215127,3.281000
3,50,20,30,No,0.319916,0.226449,3.052001
4,300,7,30,No,0.320038,0.124547,4.338005
5,300,10,30,No,0.320637,0.124547,4.368996
6,300,15,30,No,0.320903,0.113225,5.103008
7,300,20,30,No,0.320817,0.135870,5.186990
